In [0]:
%pip install great-expectations

In [0]:
import great_expectations as gx
import pandas as pd

# Load Bronze CRM customers as pandas df

In [0]:
df = spark.table("workspace.bronze.crm_cust_info").toPandas()

#Create Great Expectations workspace

In [0]:
context = gx.get_context()

data_source = context.data_sources.add_pandas("bronze.crm_cust_info")
data_asset = data_source.add_dataframe_asset("crm_cust_info")
batch_definition = data_asset.add_batch_definition_whole_dataframe("full_table")
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})


#Create expectation suite

In [0]:
suite = context.suites.add(gx.ExpectationSuite(name="crm_customers_suite"))

#Define expectations

In [0]:
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="cst_id"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(column="cst_gndr", value_set=["M", "F", "n/a", "", None]))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(column="cst_marital_status", value_set=["S", "M", "n/a", "", None]))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column="cst_id"))

# Run validation

In [0]:
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="crm_customers_validation",
        data=batch_definition,
        suite=suite
    )
)

results = validation_definition.run(batch_parameters={"dataframe": df})
print(results)

print(f"Total expectations: {results['statistics']['evaluated_expectations']}")
print(f"Passed: {results['statistics']['successful_expectations']}")
print(f"Failed: {results['statistics']['unsuccessful_expectations']}")
print(f"Success rate: {results['statistics']['success_percent']}%")

if not results['success']:
    print("\nFailed checks:")
    for r in results['results']:
        if not r['success']:
            col = r['expectation_config']['kwargs']['column']
            check = r['expectation_config']['type']
            count = r['result']['unexpected_count']
            print(f"  - {check} on '{col}': {count} violations")